<a href="https://colab.research.google.com/github/Keistkmiya/Tugas1-MachineLearning/blob/main/Tugas1_Chapter7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 7: Working with Text Data

## Setup and Library Imports

In [1]:
!pip install mglearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.4/581.4 kB 7.5 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mglearn
import warnings
from sklearn.feature_extraction.text import CountVectorizer

%matplotlib inline

## Representing Text Data as a Bag of Words

Cara paling sederhana dan paling umum digunakan untuk merepresentasikan data teks dalam machine learning adalah dengan menggunakan representasi "Bag of Words" (kantong kata).



Metode ini sangat unik karena membuang sebagian besar struktur data (seperti tata bahasa, urutan kata, dan tanda baca). Model hanya menghitung seberapa sering setiap kata muncul di dalam sebuah teks. Bayangkan kita memasukkan semua kata dari sebuah kalimat ke dalam satu kantong besar, lalu kita hitung jumlah masing-masing kata tersebut.

In [3]:
bards_words = ["The fool doth think he is wise,",
               "but the wise man knows himself to be a fool"]

vect = CountVectorizer()
vect.fit(bards_words)

print("Ukuran kosakata: {}".format(len(vect.vocabulary_)))
print("Isi kosakata:\n {}".format(vect.vocabulary_))

Ukuran kosakata: 13
Isi kosakata:
 {'the': 9, 'fool': 3, 'doth': 2, 'think': 10, 'he': 4, 'is': 6, 'wise': 12, 'but': 1, 'man': 8, 'knows': 7, 'himself': 5, 'to': 11, 'be': 0}


### Transforming Data

Setelah kita membuat kosakata (*vocabulary*) yang berisi kata-kata unik dari semua teks yang kita punya, langkah selanjutnya adalah mengubah kalimat tersebut menjadi matriks angka menggunakan metode `transform`. Angka-angka inilah yang nantinya akan dilatih oleh model machine learning kita.

In [4]:
bag_of_words = vect.transform(bards_words)

print("Representasi bag_of_words:\n{}".format(repr(bag_of_words)))
print("\nBentuk matriks padat (Dense):\n{}".format(bag_of_words.toarray()))

Representasi bag_of_words:
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 16 stored elements and shape (2, 13)>

Bentuk matriks padat (Dense):
[[0 0 1 1 1 0 1 0 0 1 1 0 1]
 [1 1 0 1 0 1 0 1 1 1 0 1 1]]


### Stopwords

Dalam pengolahan teks, ada banyak kata yang sangat sering muncul namun hampir tidak membawa informasi yang bermakna, seperti "the", "a", "is", atau "and". Kata-kata ini dikenal dengan sebutan *stopwords*. Kita dapat menyuruh `CountVectorizer` dari scikit-learn untuk mengabaikan kata-kata tersebut agar model kita bisa lebih fokus pada kata-kata yang benar-benar mendeskripsikan isi teks.

In [5]:
vect_stop = CountVectorizer(stop_words="english")
vect_stop.fit(bards_words)

print("Ukuran kosakata dengan stopwords: {}".format(len(vect_stop.vocabulary_)))
print("Isi kosakata:\n {}".format(vect_stop.vocabulary_))

Ukuran kosakata dengan stopwords: 6
Isi kosakata:
 {'fool': 1, 'doth': 0, 'think': 4, 'wise': 5, 'man': 3, 'knows': 2}


### Rescaling the Data with tf-idf

Selain membuang *stopwords*, ada teknik pembobotan yang jauh lebih canggih bernama TF-IDF (Term Frequency - Inverse Document Frequency).

Ide utamanya sangat brilian: kita memberikan bobot (nilai) yang tinggi pada suatu kata jika kata tersebut sering muncul dalam satu kalimat/dokumen tertentu, tetapi sangat jarang muncul di dokumen-dokumen lainnya secara keseluruhan. Kata yang memenuhi kriteria ini biasanya memiliki makna spesifik yang sangat mendeskripsikan kalimat tersebut. Sebaliknya, kata yang muncul di mana-mana akan langsung ditekan bobotnya menjadi sangat rendah.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
tfidf.fit(bards_words)

tfidf_matrix = tfidf.transform(bards_words)

print("Bentuk matriks tf-idf:\n{}".format(tfidf_matrix.toarray()))

Bentuk matriks tf-idf:
[[0.         0.         0.42567716 0.30287281 0.42567716 0.
  0.42567716 0.         0.         0.30287281 0.42567716 0.
  0.30287281]
 [0.36469323 0.36469323 0.         0.25948224 0.         0.36469323
  0.         0.36469323 0.36469323 0.25948224 0.         0.36469323
  0.25948224]]


### Bag-of-Words with More Than One Word (n-Grams)

Kelemahan utama dari representasi *bag-of-words* standar adalah metode ini sepenuhnya membuang urutan kata. Untuk menangkap konteks kalimat dengan lebih baik, kita bisa mengubah cara ekstraksi fitur agar tidak hanya melihat satu kata tunggal (unigram), tetapi juga melihat pasangan dua kata berurutan (bigram), tiga kata berurutan (trigram), dan seterusnya. Teknik ini secara umum disebut dengan **n-grams**.

Kita bisa menggunakan parameter `ngram_range` pada `CountVectorizer` atau `TfidfVectorizer` untuk menentukan seberapa panjang rentang kata yang ingin kita perhatikan.

In [7]:
cv_bigram = CountVectorizer(ngram_range=(2, 2)).fit(bards_words)

print("Ukuran kosakata (hanya bigrams): {}".format(len(cv_bigram.vocabulary_)))
print("Isi kosakata:\n{}".format(cv_bigram.vocabulary_))

Ukuran kosakata (hanya bigrams): 14
Isi kosakata:
{'the fool': 9, 'fool doth': 3, 'doth think': 2, 'think he': 11, 'he is': 4, 'is wise': 6, 'but the': 1, 'the wise': 10, 'wise man': 13, 'man knows': 8, 'knows himself': 7, 'himself to': 5, 'to be': 12, 'be fool': 0}


Dalam praktiknya, kita sering kali menggabungkan unigram (satu kata) dengan bigram (dua kata) sekaligus. Dengan begini, model kita tetap mengetahui frekuensi kemunculan kata tunggal, namun juga tidak kehilangan konteks dari frasa pendek yang terbentuk.

In [8]:
cv_mixed = CountVectorizer(ngram_range=(1, 2)).fit(bards_words)

print("Ukuran kosakata (unigrams dan bigrams): {}".format(len(cv_mixed.vocabulary_)))
print("Sebagian isi kosakata:\n{}".format(list(cv_mixed.vocabulary_.keys())[:10]))

Ukuran kosakata (unigrams dan bigrams): 27
Sebagian isi kosakata:
['the', 'fool', 'doth', 'think', 'he', 'is', 'wise', 'the fool', 'fool doth', 'doth think']


### Advanced Tokenization: Stemming and Lemmatization

Seringkali, satu kata dasar bisa memiliki banyak bentuk karena adanya imbuhan (misalnya: "run", "running", "runs"). Bagi model *machine learning*, kata-kata ini akan dianggap sebagai fitur yang sepenuhnya berbeda. Untuk mengatasinya, kita bisa menggunakan teknik normalisasi kata.

Ada dua metode utama:
1.  **Stemming**: Memotong akhiran kata menggunakan aturan logis sederhana (seringkali menghasilkan kata yang tidak baku).
2.  **Lemmatization**: Menggunakan kamus tata bahasa yang lengkap untuk menemukan kata dasar yang sebenarnya (lemma).

In [9]:
import nltk
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
sample_words = ["wolves", "running", "bards", "wise", "thinking"]

stemmed_words = [stemmer.stem(w) for w in sample_words]

print("Kata asli:", sample_words)
print("Hasil Stemming:", stemmed_words)

Kata asli: ['wolves', 'running', 'bards', 'wise', 'thinking']
Hasil Stemming: ['wolv', 'run', 'bard', 'wise', 'think']


### Topic Modeling with Latent Dirichlet Allocation (LDA)

Bagaimana jika kita memiliki ribuan teks dan tidak memiliki label sama sekali? Kita bisa menggunakan teknik *unsupervised learning* yang disebut *Topic Modeling*. Algoritma yang paling populer untuk ini adalah Latent Dirichlet Allocation (LDA).

LDA berusaha mencari kelompok kata (topik) yang sering muncul bersamaan di dalam berbagai dokumen. Model ini tidak mengerti makna teksnya, melainkan hanya melihat pola distribusi frekuensi kata.

In [10]:
from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(n_components=2, learning_method="batch", max_iter=25, random_state=0)
document_topics = lda.fit_transform(bag_of_words)

print("Bentuk matriks distribusi topik (dokumen x topik): {}".format(document_topics.shape))
print("Distribusi topik pada kalimat pertama: {}".format(document_topics[0]))

Bentuk matriks distribusi topik (dokumen x topik): (2, 2)
Distribusi topik pada kalimat pertama: [0.92626504 0.07373496]
